## 自定义手写数字识别 — 多模型对比

### 学习目标

1. **交互式推理体验** — 在画布上手写数字，直观感受模型如何「看到」你的笔迹
2. **理解输入格式差异** — 同一张图片，MLP 需要展平向量、CNN/ResNet/ViT 需要图像张量 `(1,28,28)`、LSTM 需要序列 `(28,28)`
3. **多架构横向对比** — 同时观察五种模型对同一输入的预测差异和置信度

### 五种模型的输入格式

| 模型 | 输入格式 | 核心操作 |
|------|----------|----------|
| MLP | `(1, 784)` 展平向量 | 全连接层逐层降维 |
| CNN | `(1, 1, 28, 28)` 图像 | 3×3 卷积核滑动扫描 |
| ResNet | `(1, 1, 28, 28)` 图像 | 卷积 + 残差跳跃连接 |
| ViT | `(1, 1, 28, 28)` 图像 | 切分为 4×4 patches → Transformer |
| LSTM | `(1, 28, 28)` 序列 | 逐行扫描，双向门控 |

> 画布 280×280，自动 LANCZOS 缩放到 28×28。**黑底白字**，与 MNIST 训练数据格式一致。

## 1. 导入依赖与模型定义

### 为什么需要定义模型结构？

推理时我们只加载 `.pth` 权重文件，但 PyTorch 需要先实例化一个**结构完全一致**的模型对象，
然后调用 `load_state_dict()` 将权重「填入」模型。如果这里的模型定义与训练时不同（层数、通道数等），加载会直接报错 `size mismatch`。

### MODEL_CONFIG 注册表用法

`MODEL_CONFIG` 是一个字典，统一管理所有模型的路径、输入格式和创建函数。**想添加新模型？只需在此注册**：

```python
# 添加新模型只需在 MODEL_CONFIG 中加一项:
MODEL_CONFIG['my_new_model'] = {
    'name': 'MyModel',              # 显示名称
    'path': '../models/xxx/best.pth', # .pth 权重文件路径
    'input_type': 'image',           # 输入格式: flat / image / sequence
    'create': lambda: NewModel(...).to(device)  # 模型构建函数
}
```

**input_type 决定数据处理方式**：
- `'flat'` → 用 `(1, 784)` 展平向量（MLP）
- `'image'` → 用 `(1, 1, 28, 28)` 图像张量（CNN/ResNet/ViT）
- `'sequence'` → 用 `(1, 28, 28)` 序列格式（LSTM）

> **约定**：以下每个类的结构必须与对应训练 notebook（02/05/06/07/08）中的定义完全相同。

In [1]:
# [1.1 导入依赖]
# matplotlib + ipympl: 交互式画布 | PIL: 高质量图像缩放 | torch: 模型推理
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from PIL import Image

# ==== 中文字体配置 ====
# matplotlib 默认不支持中文，需手动指定支持中文的字体
# 优先级：微软雅黑 > 黑体 > DejaVu Sans（英文回退）
mpl.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
mpl.rcParams['axes.unicode_minus'] = False  # 防止负号显示为方块

# ======= 超参数集中修改区 =======
# 选择要用于推理的模型（注释/删除来开关）：
PREDICT_MODELS = [
    'mlp',
    'cnn',
    'resnet',
    'vit',
    'lstm',
]

# 模型注册表：统一管理路径、构建参数
# 修改模型结构请在这里改 create 中的参数（如 hidden_dims、num_blocks 等）
MODEL_CONFIG = {
    'mlp': {
        'name': 'MLP',
        'path': '../models/mlp/best.pth',
        'input_type': 'flat',
        'create': lambda: MLP(input_dim=784, hidden_dims=[1024, 256],
                              num_classes=10, dropout=0.2).to(device)
    },
    'cnn': {
        'name': 'CNN',
        'path': '../models/cnn/best.pth',
        'input_type': 'image',
        'create': lambda: SimpleCNN(num_classes=10, dropout=0.3).to(device)
    },
    'resnet': {
        'name': 'ResNet',
        'path': '../models/resnet/best.pth',
        'input_type': 'image',
        'create': lambda: ResNet(BasicBlock, num_blocks=[2, 2, 2],
                                 num_classes=10).to(device)
    },
    'vit': {
        'name': 'ViT',
        'path': '../models/vit/best.pth',
        'input_type': 'image',
        'create': lambda: ViT(img_size=28, patch_size=4, in_channels=1,
                              num_classes=10, embed_dim=64, depth=4,
                              num_heads=4, mlp_ratio=2, dropout=0.1).to(device)
    },
    'lstm': {
        'name': 'LSTM',
        'path': '../models/lstm/best.pth',
        'input_type': 'sequence',
        'create': lambda: LSTMModel(input_dim=28, hidden_dim=128, num_layers=2,
                                    num_classes=10, dropout=0.3,
                                    bidirectional=True).to(device)
    }
}
# ================================

# ==== 设备选择 ====
# 推理对算力要求低，CPU 完全够用；有 GPU 则自动加速
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'设备: {device}')

设备: cuda


In [2]:
# [1.2 模型类定义] 五种模型结构必须与训练 notebook 完全相同，否则权重加载会失败


# 模型 1/5: MLP — 多层感知机 (输入: B×784 展平向量)
class MLP(nn.Module):
    """
    多层感知机 — 全连接层堆叠

    结构:
        Linear(784→512) → BN → ReLU → Dropout
        → Linear(512→256) → BN → ReLU → Dropout
        → Linear(256→10)

    注意:
        输出 logits 不经过 softmax，推理时手动 softmax 得到概率
    """
    def __init__(self, input_dim=784, hidden_dims=None, num_classes=10, dropout=0.2):
        super(MLP, self).__init__()
        if hidden_dims is None:
            hidden_dims = [512, 256]

        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        """
        Args:
            x: (B, 784) 展平后的灰度图像向量
        Returns:
            (B, 10) logits
        """
        return self.net(x)


# 模型 2/5: SimpleCNN — 卷积网络 (输入: B×1×28×28 图像)
class SimpleCNN(nn.Module):
    """
    轻量 CNN — 卷积 Block + 全连接分类头

    每个 Conv Block:
        N×[Conv2d → BN → ReLU] → MaxPool2d(2) → Dropout2d

    注意:
        flatten_dim 自动计算，无需手动改代码
    """
    def __init__(self, num_classes=10, dropout=0.3,
                 conv_channels=None, conv_per_block=2,
                 kernel_size=3, fc_hidden=128,
                 in_channels=1, input_size=28):
        super(SimpleCNN, self).__init__()
        if conv_channels is None:
            conv_channels = [32, 64]

        in_ch = in_channels
        self.conv_blocks = nn.ModuleList()
        for out_ch in conv_channels:
            layers = []
            for j in range(conv_per_block):
                layers.extend([
                    nn.Conv2d(in_ch if j == 0 else out_ch, out_ch,
                              kernel_size=kernel_size, padding=kernel_size // 2),
                    nn.BatchNorm2d(out_ch),
                    nn.ReLU(),
                ])
            layers.extend([nn.MaxPool2d(2), nn.Dropout2d(dropout)])
            self.conv_blocks.append(nn.Sequential(*layers))
            in_ch = out_ch

        final_spatial = input_size // (2 ** len(conv_channels))
        flatten_dim = conv_channels[-1] * final_spatial * final_spatial

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flatten_dim, fc_hidden),
            nn.BatchNorm1d(fc_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fc_hidden, num_classes)
        )

    def forward(self, x):
        """
        Args:
            x: (B, 1, 28, 28) 图像张量
        Returns:
            (B, 10) logits
        """
        for block in self.conv_blocks:
            x = block(x)
        return self.classifier(x)


# 模型 3/5: ResNet — 残差网络 (输入: B×1×28×28 图像)
class BasicBlock(nn.Module):
    """
    ResNet 基础残差块

    主路: Conv3×3 → BN → ReLU → Conv3×3 → BN
    短路: 恒等映射 (或 1×1 卷积对齐维度)
    输出: ReLU(主路 + 短路)  ← 核心: F(x) + x

    注意:
        expansion = 1 (BasicBlock 输出通道 = out_channels)
        对比 Bottleneck (ResNet-50/101) 的 expansion=4
    """
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        """
        Args:
            x: (B, C_in, H, W) 输入特征图
        Returns:
            (B, C_out, H', W') 残差块输出
        """
        identity = self.shortcut(x)
        out = nn.ReLU()(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        return nn.ReLU()(out)


class ResNet(nn.Module):
    """
    轻量 ResNet，为小尺寸图像（28×28）设计

    整体结构: conv1 → [layer1] → [layer2] → [layer3] → avgpool → fc

    可调参数:
        block:            残差块类型（BasicBlock）
        num_blocks:       每层 block 数量，如 [2,2,2] 共 3 层各 2 个 block
        channel_list:     每层输出通道数，默认 [32, 64, 128]
                          注意: 长度必须 = len(num_blocks)
        in_planes:        初始卷积层输出通道数
    """
    def __init__(self, block, num_blocks, num_classes=10,
                 in_planes=32, channel_list=None):
        super(ResNet, self).__init__()
        if channel_list is None:
            channel_list = [32, 64, 128]
        self.in_channels = in_planes
        self.num_layers = len(channel_list)
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, in_planes, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(in_planes), nn.ReLU()
        )
        for i, (ch, nb) in enumerate(zip(channel_list, num_blocks)):
            stride = 1 if i == 0 else 2
            layer = self._make_layer(block, ch, nb, stride=stride)
            setattr(self, f'layer{i + 1}', layer)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(channel_list[-1], num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        """
        构建一个残差层（多个 BasicBlock 堆叠）

        约定:
            第一个 block 负责通道/尺寸变换
            后续 block 通道/尺寸不变，纯加深网络
        """
        layers = [block(self.in_channels, out_channels, stride)]
        self.in_channels = out_channels  # 更新通道数供后续 block 使用
        for _ in range(1, num_blocks):
            layers.append(block(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        """
        Args:
            x: (B, 1, 28, 28) 图像
        Returns:
            (B, 10) logits
        """
        # 1. 初始卷积: (B,1,28,28) → (B,32,28,28)
        x = self.conv1(x)
        for i in range(1, self.num_layers + 1):
            x = getattr(self, f'layer{i}')(x)
        x = self.avgpool(x)
        return self.fc(torch.flatten(x, 1))


# 模型 4/5: ViT — Vision Transformer (输入: B×1×28×28 图像)
class PatchEmbedding(nn.Module):
    """
    将图片切分为 patch 并做线性投影

    28×28, patch_size=4 → 7×7 = 49 个 patch
    每个 patch 被投影为 embed_dim=64 维向量

    用 Conv2d(stride=patch_size) 实现，比手动切分更高效
    """
    def __init__(self, img_size=28, patch_size=4, in_channels=1, embed_dim=64):
        super(PatchEmbedding, self).__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        """
        Args:
            x: (B, C, H, W) 图像
        Returns:
            (B, num_patches, embed_dim) patch 序列
        """
        x = self.proj(x).flatten(2).transpose(1, 2)
        return x


class ViT(nn.Module):
    """
    轻量 Vision Transformer，为 MNIST 设计

    数据流:
        28×28 → PatchEmbedding → 49 patches × 64 dim
        → + [CLS] token + Position Embedding → 50 tokens
        → Transformer Encoder × 4 (Self-Attention + FFN)
        → 取 [CLS] 输出 → LayerNorm → Linear(64→10)

    关键设计:
        - Pre-LN (norm_first=True): 训练更稳定
        - GELU 激活: Transformer 标配
        - Class Token: 聚合全局信息的可学习向量
    """
    def __init__(self, img_size=28, patch_size=4, in_channels=1,
                 num_classes=10, embed_dim=64, depth=4, num_heads=4,
                 mlp_ratio=2, dropout=0.1):
        super(ViT, self).__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=embed_dim * mlp_ratio,
            dropout=dropout, activation='gelu',
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth,
                                               enable_nested_tensor=False)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        """
        Args:
            x: (B, 1, 28, 28) 图像
        Returns:
            (B, 10) logits
        """
        B = x.shape[0]
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = self.pos_drop(x + self.pos_embed)
        x = self.transformer(x)
        return self.head(self.norm(x[:, 0]))


# 模型 5/5: LSTM — 双向长短期记忆 (输入: B×28×28 序列)
class LSTMModel(nn.Module):
    """
    双向 LSTM + 全连接分类头

    核心思想:
        将 28×28 图片视为 28 个时间步的序列，每步输入一行 28 像素
        双向扫描（正向 1→28 + 反向 28→1），拼接隐状态后分类

    架构:
        LSTM(28→128, 2层, 双向) → concat(h_fwd, h_bwd)→256
        → Dropout → FC(256→64) → ReLU → Dropout → FC(64→10)

    注意:
        batch_first=True: 输入 (B, seq, feat) 而非默认的 (seq, B, feat)
        bidirectional: 隐状态维度翻倍（128×2=256）
    """
    def __init__(self, input_dim=28, hidden_dim=128, num_layers=2,
                 num_classes=10, dropout=0.3, bidirectional=True):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        self.lstm = nn.LSTM(
            input_size=input_dim, hidden_size=hidden_dim,
            num_layers=num_layers, batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        fc_input_dim = hidden_dim * self.num_directions
        self.fc = nn.Sequential(
            nn.Linear(fc_input_dim, 64),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        """
        Args:
            x: (B, 28, 28) 序列
        Returns:
            (B, 10) logits
        """
        lstm_out, (h_n, c_n) = self.lstm(x)
        if self.bidirectional:
            h_forward  = h_n[-2, :, :]
            h_backward = h_n[-1, :, :]
            h_last = torch.cat([h_forward, h_backward], dim=1)
        else:
            h_last = h_n[-1, :, :]
        return self.fc(self.dropout(h_last))

# 根据 PREDICT_MODELS 筛选要加载的模型
active_config = {k: v for k, v in MODEL_CONFIG.items() if k in PREDICT_MODELS}
print('已注册模型: {total}, 本次评估: {active}'.format(
    total=list(MODEL_CONFIG.keys()),
    active=list(active_config.keys())
))

已注册模型: ['mlp', 'cnn', 'resnet', 'vit', 'lstm'], 本次评估: ['mlp', 'cnn', 'resnet', 'vit', 'lstm']


## 2. 书写数字

> 在下方 280×280 画布上**按住鼠标拖动**来写一个数字。黑色背景，白色笔迹（与 MNIST 一致）。
> 写完一个数字后运行下一个 cell 进行识别。

In [4]:
# [2. 交互画布] 280×280 鼠标绘制 | 事件驱动: press→motion→release | 圆形笔触点
# 启用交互式后端（支持鼠标事件绑定）
%matplotlib notebook

from matplotlib.widgets import Button

# ==== 画布初始化 ====
CANVAS_SIZE = 280     # 画布尺寸（像素）
# 全零数组 = 黑色画布，uint8 像素范围 [0, 255]
canvas = np.zeros((CANVAS_SIZE, CANVAS_SIZE), dtype=np.uint8)

# 用字典追踪绘制状态（闭包中可变，不需要 global 声明）
drawing = {'active': False}

# ==== 鼠标事件处理函数 ====
def on_press(event):
    """鼠标按下：激活绘制状态，立即画第一个点"""
    if event.inaxes == ax_draw:       # 只在画布区域内响应
        drawing['active'] = True
        draw_point(event)             # 按下时也画一个点（避免只能拖动画）

def on_release(event):
    """鼠标松开：停止绘制"""
    drawing['active'] = False

def on_motion(event):
    """鼠标拖动：连续绘制（仅当 pressed 状态为 True）"""
    if drawing['active'] and event.inaxes == ax_draw:
        draw_point(event)

def draw_point(event):
    """
    在画布上绘制一个圆形笔触点

    实现: 以鼠标坐标为中心，半径为 r 的圆形区域填充 255（白色）
    用 dx² + dy² ≤ r² 判断是否在圆内
    """
    x, y = int(event.xdata), int(event.ydata)  # 鼠标在图中的坐标
    r = 8  # 笔画半径（像素）

    # 遍历以 (x,y) 为中心的 (2r+1)×(2r+1) 方形区域
    for dx in range(-r, r + 1):
        for dy in range(-r, r + 1):
            if dx**2 + dy**2 <= r**2:          # 圆形判断
                nx, ny = x + dx, y + dy
                # 边界检查：只在画布内绘制
                if 0 <= nx < CANVAS_SIZE and 0 <= ny < CANVAS_SIZE:
                    canvas[ny, nx] = 255       # 设为白色

    # 更新显示：刷新 imshow 的数据并重绘画布
    draw_img.set_data(canvas)
    fig.canvas.draw_idle()

def clear_canvas(event):
    """清除按钮回调：将画布全部重置为 0"""
    canvas[:] = 0
    draw_img.set_data(canvas)
    fig.canvas.draw_idle()

# ==== 创建带按钮的布局 ====
fig = plt.figure(figsize=(5, 5.5))
# 画布放在上半部分
ax_draw = fig.add_axes([0.1, 0.18, 0.8, 0.78])
draw_img = ax_draw.imshow(canvas, cmap='gray', vmin=0, vmax=255)
ax_draw.set_title('在此画布上写一个数字（按住鼠标拖动）', fontsize=12)
ax_draw.axis('off')

# 清除按钮放在下半部分
ax_clear = fig.add_axes([0.35, 0.03, 0.3, 0.08])
btn_clear = Button(ax_clear, '清除重写', color='lightgray', hovercolor='tomato')
btn_clear.on_clicked(clear_canvas)

# ==== 绑定鼠标事件到画布 ====
# mpl_connect 将事件与处理函数关联
fig.canvas.mpl_connect('button_press_event', on_press)      # 按下
fig.canvas.mpl_connect('button_release_event', on_release)  # 松开
fig.canvas.mpl_connect('motion_notify_event', on_motion)    # 拖动

plt.show()
print('请在画布上书写，点击「清除重写」按钮可清空画布')

<IPython.core.display.Javascript object>

请在画布上书写，点击「清除重写」按钮可清空画布


## 3. 预处理

### 为什么需要预处理？

1. **尺寸缩放**: 画布 280×280 → MNIST 标准的 28×28。用 LANCZOS 算法保留笔画质量
2. **归一化**: 像素值 / 255 → [0, 1]，与训练数据范围一致
3. **格式转换**: 同一缩放结果生成三种格式供不同模型使用

| 格式 | 形状 | 使用者 |
|------|------|--------|
| flat 展平 | `(1, 784)` | MLP |
| image 图像 | `(1, 1, 28, 28)` | CNN / ResNet / ViT |
| sequence 序列 | `(1, 28, 28)` | LSTM |

In [5]:
# [3. 预处理] PIL LANCZOS 缩放 280→28 | 归一化 | 三种输入格式并行生成 | 对比展示

# ==== 1. 缩放: 280×280 → 28×28 ====
# PIL Image 从 numpy 数组创建灰度图
img_pil = Image.fromarray(canvas, mode='L')                         # mode='L' = 8-bit 灰度
# LANCZOS: 高质量重采样算法，保留笔画边缘清晰度
img_28 = img_pil.resize((28, 28), Image.Resampling.LANCZOS)

# ==== 2. 归一化到 [0, 1] ====
img_array = np.array(img_28, dtype=np.float32)                     # uint8 → float32
img_normalized = img_array / 255.0                                  # 归一化 [0,255]→[0,1]

# ==== 3. 三种输入格式 ====
# flat:       (784,) → (1, 784)          MLP 用展平向量
# image:      (28,28) → (1, 1, 28, 28)   CNN/ResNet/ViT 用图像格式
# sequence:   (28,28) → (1, 28, 28)      LSTM 用序列格式
input_flat     = torch.tensor(img_normalized.reshape(1, 784), dtype=torch.float32).to(device)
input_image    = torch.tensor(img_normalized.reshape(1, 1, 28, 28), dtype=torch.float32).to(device)
input_sequence = torch.tensor(img_normalized.reshape(1, 28, 28), dtype=torch.float32).to(device)

# ==== 4. 对比展示: 原始画布 vs 缩放后 ====
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 3))
ax1.imshow(canvas, cmap='gray')
ax1.set_title('原始画布 (280×280)', fontsize=10)
ax1.axis('off')
ax2.imshow(img_28, cmap='gray')
ax2.set_title('缩放后 (28×28)', fontsize=10)
ax2.axis('off')
plt.tight_layout()
plt.show()

# 统计非零像素占比（检查是否写得太淡）
print(f'非零像素占比: {(img_array > 0).mean():.2%}')
print(f'输入形状: flat={tuple(input_flat.shape)}, image={tuple(input_image.shape)}, seq={tuple(input_sequence.shape)}')

<IPython.core.display.Javascript object>

非零像素占比: 26.02%
输入形状: flat=(1, 784), image=(1, 1, 28, 28), seq=(1, 28, 28)


## 4. 多模型推理对比

### 推理流程

```
缩放后的 28×28 图片
    │
    ├─→ flat 格式  → MLP    → logits → softmax → 概率分布
    ├─→ image 格式 → CNN    → logits → softmax → 概率分布
    ├─→ image 格式 → ResNet → logits → softmax → 概率分布
    ├─→ image 格式 → ViT    → logits → softmax → 概率分布
    └─→ seq 格式   → LSTM   → logits → softmax → 概率分布
                                    ↓
                          水平柱状图对比 Top-1 预测
```

> 自动加载已训练权重，所有模型对同一张手写图片预测，横向对比概率分布。

In [6]:
# [4.1 加载模型权重] 遍历 active_config 自动加载 | 跳过未训练模型 | 显示加载状态
import os

loaded_models = {}  # {key: model_instance}
print('加载模型权重...\n')

for key, cfg in active_config.items():
    # 检查权重文件是否存在
    if not os.path.exists(cfg['path']):
        print(f'  [{cfg["name"]:>6s}] SKIP — 权重文件不存在: {cfg["path"]}')
        continue

    try:
        # 1. 通过 lambda 创建模型实例
        model = cfg['create']()
        # 2. 加载权重（map_location=device 确保权重加载到正确设备）
        model.load_state_dict(torch.load(cfg['path'], map_location=device))
        # 3. 切换为评估模式（关闭 Dropout，冻结 BN 统计量）
        model.eval()
        loaded_models[key] = model
        params = sum(p.numel() for p in model.parameters())
        print(f'  [{cfg["name"]:>6s}] OK — {params:,} 参数')
    except Exception as e:
        print(f'  [{cfg["name"]:>6s}] FAIL — {e}')

if not loaded_models:
    print('\n未加载任何模型！请先运行训练 notebook (02, 05, 06, 07, 08)。')
else:
    print(f'\n已加载 {len(loaded_models)} 个模型: {list(loaded_models.keys())}')

加载模型权重...

  [   MLP] OK — 1,071,370 参数
  [   CNN] OK — 468,458 参数
  [ResNet] OK — 696,042 参数
  [   ViT] OK — 139,018 参数
  [  LSTM] OK — 574,154 参数

已加载 5 个模型: ['mlp', 'cnn', 'resnet', 'vit', 'lstm']


In [7]:
# [4.2 多模型推理] 所有模型同时预测 | 水平柱状图(Top-1红色高亮) | 文本输出 Top-3

results = {}  # {key: {'pred': int, 'probs': np.ndarray, 'name': str}}

# ==== 遍历所有已加载模型执行推理 ====
for key, model in loaded_models.items():
    cfg = MODEL_CONFIG[key]

    with torch.no_grad():  # 禁用梯度计算（推理不需要梯度）
        # ==== 根据模型类型选择对应的输入格式 ====
        if cfg['input_type'] == 'flat':
            logits = model(input_flat)          # MLP: (1, 784) → (1, 10)
        elif cfg['input_type'] == 'sequence':
            logits = model(input_sequence)      # LSTM: (1, 28, 28) → (1, 10)
        else:
            logits = model(input_image)         # CNN/ResNet/ViT: (1, 1, 28, 28) → (1, 10)

        # softmax 将 logits 转为概率分布 [0,1]，和为 1
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred = torch.argmax(logits, dim=1).item()  # 取最大概率的类别
        results[key] = {'pred': pred, 'probs': probs, 'name': cfg['name']}

# ==== 可视化：压缩后图像 + 每个模型的水平柱状图 ====
n_models = len(results)
if n_models == 0:
    print('没有可展示的结果。')
else:
    # 布局: 第 1 列显示缩放后 28×28，后面 n_models 列显示各模型预测
    n_cols = 1 + n_models
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4.5))
    if n_cols == 1:
        axes = [axes]

    # 第 1 列: 显示缩放后的 28×28 图像（模型实际看到的）
    ax_img = axes[0]
    ax_img.imshow(img_28, cmap='gray')
    ax_img.set_title('模型输入 (28×28)', fontsize=10, fontweight='bold')
    ax_img.axis('off')

    # 后面的列: 每个模型的预测柱状图
    for ax, (key, result) in zip(axes[1:], results.items()):
        y_pos = np.arange(10)  # 0~9 十个类别

        # 配色: 预测类别红色高亮，其余蓝色
        colors = ['#1f77b4'] * 10
        colors[result['pred']] = '#d62728'

        # 水平柱状图（barh），y 轴是类别，x 轴是概率
        ax.barh(y_pos, result['probs'], color=colors, alpha=0.8)
        ax.set_yticks(y_pos)
        ax.set_yticklabels([str(i) for i in range(10)])
        ax.invert_yaxis()  # 0 放在最上面
        ax.set_xlim(0, 1)
        ax.set_xlabel('Probability')
        ax.set_title(f"{result['name']}: {result['pred']}",
                     fontsize=14, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)

    fig.suptitle('多模型预测对比', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

    # ==== 文本输出：每个模型的 Top-3 预测 ====
    print('\n' + '=' * 65)
    print(f'{"Model":<10} {"Pred":<6} {"Confidence":<12} {"Top-3 (类别:概率)"}')
    print('=' * 65)
    for key, result in results.items():
        # np.argsort 返回排序后的索引，[-3:] 取最大的 3 个，[::-1] 从大到小
        top3_idx = np.argsort(result['probs'])[-3:][::-1]
        top3_str = ' | '.join([f'{d}:{result["probs"][d]:.3f}' for d in top3_idx])
        print(f'{result["name"]:<10} {result["pred"]:<6} {result["probs"][result["pred"]]:.4f}       {top3_str}')
    print('=' * 65)

    # ==== 一致性检查 ====
    preds = [r['pred'] for r in results.values()]
    models_list = [r['name'] for r in results.values()]
    if len(set(preds)) == 1:
        print(f'\n全部一致 — 所有模型都预测为 {preds[0]}')
    else:
        print(f'\n模型存在分歧:')
        for i, (m, p) in enumerate(zip(models_list, preds)):
            print(f'  {m}: {p}')

<IPython.core.display.Javascript object>


Model      Pred   Confidence   Top-3 (类别:概率)
MLP        8      0.7106       8:0.711 | 5:0.097 | 0:0.096
CNN        5      0.9624       5:0.962 | 0:0.019 | 9:0.006
ResNet     6      0.4898       6:0.490 | 9:0.337 | 0:0.066
ViT        2      0.7833       2:0.783 | 5:0.200 | 1:0.012
LSTM       5      0.9522       5:0.952 | 3:0.032 | 8:0.011

模型存在分歧:
  MLP: 8
  CNN: 5
  ResNet: 6
  ViT: 2
  LSTM: 5


## 5. 重新绘制

> 运行下方 cell 清空画布，然后回到第 2 节重新书写。

In [22]:
# [5. 清空画布] canvas.fill(0) 将所有像素重置为 0（黑色）
# 注意: 只在本次运行中清空 numpy 数组，需回到第 2 节重新执行以刷新显示
canvas.fill(0)
print('画布已清空。回到 "2. 书写数字" 重新绘制。')

画布已清空。回到 "2. 书写数字" 重新绘制。


---

## 6. 进一步学习

- 进入 `03_evaluation.ipynb` 了解如何科学评估模型性能
- 进入 `09_knowledge_distillation.ipynb` 学习知识蒸馏
- 尝试修改 MODEL_CONFIG 添加自定义模型